In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q nlpaug transformers torch sacremoses


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.5/410.5 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 45.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import torch
import random
from transformers import pipeline
from tqdm import tqdm

In [ ]:
# 1. Load your base dataset with quoting guardrails
df = pd.read_csv(
    "/content/drive/MyDrive/AIE_Project/intent_classification_datasets/data_new.csv",
    quoting=0,
    on_bad_lines='skip'
)
df.columns = df.columns.str.strip()

# Drop any accidental completely blank rows
df = df.dropna(subset=['text', 'label'])

print(f"Original baseline dataset size: {len(df)} rows.")

# 2. Assign execution hardware acceleration
device = 0 if torch.cuda.is_available() else -1

print("Loading text augmentation model...")
mask_filler = pipeline("fill-mask", model="distilbert-base-uncased", device=device)

# --- PRODUCTION TUNING PARAMETERS ---
TARGET_ROWS_PER_CLASS = 1000  # Enforces a perfect final distribution balance
MAX_ATTEMPTS = 50            # Dynamic ceiling limit to avoid infinite loops

# High-risk business action keywords that the model must NEVER mask or replace
INTENT_ANCHOR_WORDS = {
    "refund", "money", "back", "cancel", "return", "reimburse", "payment",
    "invoice", "receipt", "charge", "shipping", "address", "delivery", "arrive",
    "carrier", "broken", "damaged", "defective", "glitch", "crash", "suicide",
    "harm", "crisis", "helpline", "password", "login", "logout", "account"
}

augmented_records = []

# 3. Dynamic Balancing Generation Loop grouped strictly by intent category
print("\nGenerating balanced synthetic variants...")
for label, group in df.groupby("label"):
    print(f"Processing intent group: [{label}]")

    # Always keep the original real-world ground-truth rows first
    class_records = group[["text", "label"]].to_dict(orient="records")

    # FIXED: Added the proper .str accessor to the strip function to fix the AttributeError
    existing_texts = set(group["text"].str.lower().str.strip())

    attempts = 0
    # Dynamic balancing guardrail loop
    while len(class_records) < TARGET_ROWS_PER_CLASS and attempts < MAX_ATTEMPTS:
        attempts += 1

        for _, row in group.iterrows():
            if len(class_records) >= TARGET_ROWS_PER_CLASS:
                break

            text = str(row['text'])
            words = text.split()

            if len(words) < 3:
                continue

            # Filter indices to prevent replacing critical anchor keywords
            valid_mask_indices = [
                i for i, word in enumerate(words)
                if word.lower().strip("?,.!") not in INTENT_ANCHOR_WORDS
            ]

            # Fallback if the sentence is completely packed with anchor keywords
            if not valid_mask_indices:
                valid_mask_indices = list(range(len(words)))

            # Pick a safe random index to mask
            mask_idx = random.choice(valid_mask_indices)

            masked_words = words.copy()
            masked_words[mask_idx] = "[MASK]"
            masked_sentence = " ".join(masked_words)

            try:
                # Generate variations
                predictions = mask_filler(masked_sentence, top_k=5)
                if not isinstance(predictions, list):
                    predictions = [predictions]

                for pred in predictions:
                    variant_text = " ".join(pred['sequence'].split())
                    variant_clean = variant_text.lower().strip()

                    # Prevent identical duplicates from eating dataset rows
                    if variant_clean not in existing_texts:
                        class_records.append({"text": variant_text, "label": label})
                        existing_texts.add(variant_clean)

                        if len(class_records) >= TARGET_ROWS_PER_CLASS:
                            break
            except Exception:
                pass

    augmented_records.extend(class_records)

# 4. Compile and verify dataset structure integrity
df_augmented = pd.DataFrame(augmented_records)
df_augmented.drop_duplicates(subset=['text'], inplace=True)

# 5. Save the output back to your drive folder
output_path = "/content/drive/MyDrive/AIE_Project/intent_classification_datasets/data_new_augmented_dataset.csv"
df_augmented.to_csv(output_path, index=False)

print("\n🚀 Balanced Expansion Complete!")
print(f"New dataset size: {len(df_augmented)} rows total.")
print(f"Saved to: '{output_path}'")
print("\nFinal verified distribution counts per intent class:")
print(df_augmented['label'].value_counts())


Original baseline dataset size: 221 rows.
Loading text augmentation model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]


Generating balanced synthetic variants...
Processing intent group: [complaint]


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Processing intent group: [general_question]
Processing intent group: [logistics]
Processing intent group: [payment]
Processing intent group: [product_inquiry]
Processing intent group: [refund]
Processing intent group: [self_harm_or_suicide_risk]
Processing intent group: [technical_support]
Processing intent group: [unknown]

🚀 Balanced Expansion Complete!
New dataset size: 7936 rows total.
Saved to: '/content/drive/MyDrive/AIE_Project/intent_classification_datasets/data_new_augmented_dataset.csv'

Final verified distribution counts per intent class:
label
complaint                    1000
technical_support            1000
self_harm_or_suicide_risk    1000
general_question              980
refund                        918
product_inquiry               886
payment                       763
unknown                       704
logistics                     685
Name: count, dtype: int64
